In [2]:
import os
import sys
import numpy as np
import gymnasium as gym
from sumo_rl import SumoEnvironment
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
import pickle

In [3]:
os.environ['SUMO_HOME'] = r'C:\Program Files (x86)\Eclipse\Sumo' 
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    sys.exit("SUMO_HOME not found.")

BASE_PATH = r'C:\Users\VICTUS\Documents\AI_Project\Intelligent-Traffic-Signal-Control-using-Reinforcement-Learning\Temp\VS_code'

def get_valid_path(base, filename):
    path = os.path.join(base, filename)
    if os.path.exists(path):
        return path
    return None

# Downloading these files from the "Intersection Model" folder is required

# windows:
#NET_FILE = get_valid_path(BASE_PATH, 'cross.net.xml')
#ROU_FILE = get_valid_path(BASE_PATH, 'cross.rou.xml')

#linux:
NET_FILE = "./cross.net.xml"
ROU_FILE = "./cross.rou.xml"

if not NET_FILE or not ROU_FILE:
    print(f"CRITICAL ERROR: 'cross.net.xml' or 'cross.rou.xml' not found")
    sys.exit()


In [4]:
class SARSAAgent:
    """
    SARSA (State-Action-Reward-State-Action) reinforcement learning agent.
    
    This agent uses a Q-table to learn optimal policies for traffic signal control.
    """
    def __init__(self, action_size):
        self.q_table = {}
        self.lr, self.gamma, self.epsilon = 0.1, 0.95, 1.0
        self.action_size = action_size

    def get_state_key(self, state):
        return tuple(np.round(state, 1))

    def choose_action(self, state):
        state_key = self.get_state_key(state)
        if state_key not in self.q_table:
            self.q_table[state_key] = np.zeros(self.action_size)
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_size)
        return np.argmax(self.q_table[state_key])

    def learn(self, s, a, r, s_next, a_next):
        s_k, sn_k = self.get_state_key(s), self.get_state_key(s_next)
        if sn_k not in self.q_table: self.q_table[sn_k] = np.zeros(self.action_size)
        predict = self.q_table[s_k][a]
        target = r + self.gamma * self.q_table[sn_k][a_next]
        self.q_table[s_k][a] += self.lr * (target - predict)
    
    def decay_epsilon(self, decay_rate=0.995, min_epsilon=0.01):
        """Decay the exploration rate."""
        self.epsilon = max(min_epsilon, self.epsilon * decay_rate)

    def save_model(self, filepath="sarsa_qtable.pkl"):
        """Saves the Q-table to a file."""
        with open(filepath, 'wb') as f:
            pickle.dump(self.q_table, f)
        print(f"Model successfully saved to: {filepath}")

    def load_model(self, filepath="sarsa_qtable.pkl"):
        """Loads the Q-table from a file."""
        if os.path.exists(filepath):
            with open(filepath, 'rb') as f:
                self.q_table = pickle.load(f)
            print(f"Model successfully loaded from: {filepath}")
        else:
            print(f"Warning: No saved model found at {filepath}")

In [5]:
def evaluate_model(agent, net_file, rou_file, tripinfo_file="tripinfo.xml"):
    """
    Evaluates the trained agent and generates a tripinfo.xml file.
    """
    print(f"\n--- Starting Evaluation ---")
    print(f"Generating trip info file at: {tripinfo_file}")
    
    # Initialize a fresh environment specifically for evaluation
    eval_env = SumoEnvironment(
        net_file=net_file, 
        route_file=rou_file,
        use_gui=False, # Set to True if you want to watch the trained agent
        num_seconds=3600,
        single_agent=True,
        additional_sumo_cmd=f"--tripinfo-output {tripinfo_file}" # Tells SUMO to generate the XML
    )
    
    state, info = eval_env.reset()
    done = False
    total_eval_reward = 0
    
    # Save the original epsilon and set it to 0 for pure exploitation (no random actions)
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0
    
    while not done:
        action = agent.choose_action(state)
        next_state, reward, terminated, truncated, info = eval_env.step(action)
        
        done = terminated or truncated
        state = next_state
        total_eval_reward += reward
        
    # Clean up and restore
    agent.epsilon = original_epsilon
    eval_env.close()
    
    print(f"Evaluation completed! Total Reward: {total_eval_reward}")
    print(f"Trip info successfully saved to: {os.path.abspath(tripinfo_file)}\n")


In [15]:
# Initialize environment with the 4-way intersection files
env = SumoEnvironment(
    net_file=NET_FILE, 
    route_file=ROU_FILE,
    use_gui=False,
    num_seconds=3600,
    single_agent=True,
    sumo_warnings = False
)

 Retrying in 1 seconds


Step #0.00 (0ms ?*RT. ?UPS, TraCI: 5ms, vehicles TOT 0 ACT 0 BUF 0)                      


In [16]:
# Environment will now detect the traffic light phases from cross.net.xml
action_size = env.action_space.n
agent = SARSAAgent(action_size=action_size)

# Initialize TensorBoard writer
writer = SummaryWriter(log_dir='./sarsa_tensorboard')

print("Starting SARSA Training...")
episodes = 50
reward_history = []

for e in tqdm(range(episodes), desc="Training Episodes"):
    state, info = env.reset()
    action = agent.choose_action(state)
    total_reward = 0
    done = False
    
    while not done:
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        next_action = agent.choose_action(next_state)
        agent.learn(state, action, reward, next_state, next_action)
        
        state = next_state
        action = next_action
        total_reward += reward
    
    reward_history.append(total_reward)
    # Log to TensorBoard
    writer.add_scalar('Reward/Episode', total_reward, e + 1)
    writer.add_scalar('Epsilon', agent.epsilon, e + 1)
    # Decay epsilon
    agent.decay_epsilon()

writer.close()
print("Training completed successfully.")

Starting SARSA Training...


Training Episodes:   0%|                                                                                                                   | 0/50 [00:00<?, ?it/s]

 Retrying in 1 seconds


Training Episodes:   2%|██▏                                                                                                        | 1/50 [00:21<17:18, 21.19s/it]Warning: Environment variable SUMO_HOME is not set properly, disabling XML validation. Set 'auto' or 'always' for web lookups.


Step #3600.00 (1ms ~= 1000.00*RT, ~47000.00UPS, TraCI: 21ms, vehicles TOT 1311 ACT 47 BUF 
 Retrying in 1 seconds


...

Step #3600.00 (1ms ~= 1000.00*RT, ~43000.00UPS, TraCI: 37ms, vehicles TOT 1311 ACT 43 BUF 
 Retrying in 1 seconds


Training Episodes:  98%|███████████████████████████████████████████████████████████████████████████████████████████████████████▉  | 49/50 [19:23<00:25, 25.98s/it]Warning: Environment variable SUMO_HOME is not set properly, disabling XML validation. Set 'auto' or 'always' for web lookups.


Step #3600.00 (1ms ~= 1000.00*RT, ~44000.00UPS, TraCI: 35ms, vehicles TOT 1311 ACT 44 BUF 
 Retrying in 1 seconds


Training Episodes: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [19:49<00:00, 23.78s/it]

Training completed successfully.


In [17]:
env.close()

Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 18464ms, vehicles TOT 1311 ACT 45 BUF 0)            


In [18]:
agent.save_model()

Model successfully saved to: sarsa_qtable.pkl


In [19]:
evaluate_model(agent, NET_FILE, ROU_FILE, tripinfo_file="tripinfo_eval.xml")


--- Starting Evaluation ---
Generating trip info file at: tripinfo_eval.xml
 Retrying in 1 seconds


Step #0.00 (0ms ?*RT. ?UPS, TraCI: 6ms, vehicles TOT 0 ACT 0 BUF 0)                      
 Retrying in 1 seconds


Step #3600.00 (0ms ?*RT. ?UPS, TraCI: 27ms, vehicles TOT 1311 ACT 41 BUF 0)               
Evaluation completed! Total Reward: 2.7755575615628914e-17
Trip info successfully saved to: /home/ash/projects/Intelligent-Traffic-Signal-Control-using-Reinforcement-Learning/SARSA/tripinfo_eval.xml

